# Pairwise label consistency

Minimal notebook for comparing two pairwise ranking exports with Cohen's kappa.

In [ ]:
import json
from pathlib import Path

import pandas as pd

base = Path("data/label-consistency-test") if Path("data/label-consistency-test").exists() else Path("notebooks/data/label-consistency-test")


def read_export(folder):
    rows = []
    for path in sorted((base / folder).glob("*")):
        row = json.loads(path.read_text())
        if not row.get("result"):
            continue
        left = Path(row["task"]["data"]["left"].split("://")[-1]).name
        right = Path(row["task"]["data"]["right"].split("://")[-1]).name
        pair = tuple(sorted((left, right)))
        picked = left if row["result"][0]["value"]["selected"] == "left" else right
        rows.append({"pair": pair, f"choice_{folder}": int(picked == pair[1])})
    return pd.DataFrame(rows)


matching = read_export("ranks_1").merge(read_export("ranks_2"), on="pair")
matching.head()

,pair,choice_ranks_1,choice_ranks_2
0,(25-11-02 19-14-55 5695-00.05.54.650-00.05.58....,1,1
1,(25-11-02 19-14-55 5695-00.05.54.650-00.05.58....,1,1
2,(25-11-02 19-14-55 5695-00.05.54.650-00.05.58....,1,1
3,(25-11-02 19-14-55 5695-00.05.54.650-00.05.58....,1,0
4,(25-11-02 19-14-55 5695-00.05.54.650-00.05.58....,1,1


In [5]:
from sklearn.metrics import cohen_kappa_score

binary = matching[["choice_ranks_1", "choice_ranks_2"]].dropna().astype(int)

cohen_kappa_score(binary["choice_ranks_1"], binary["choice_ranks_2"])


0.7747747747747747